# 03 — Run AIFS forecast on Modal

Dispatches a 7-day AIFS forecast to a Modal GPU (~$0.05, ~2 min).
Initial conditions must already be ingested (see `03-ingest-ic.ipynb`).
Results are stored in an icechunk/Zarr repository in Tigris object storage.

**Stack**: `aifs-modal` → Modal (A100) → icechunk on Tigris

In [ ]:
import datetime as dt
import pathlib
import time

from aifs_modal import app, run_forecast

In [ ]:
storm_name = "claudia"
init_date_str = "2025110700"  # %Y%m%d%H — D-1 from ERA5 peak (2025-11-08)
lead_time = 168  # hours = 7 days
n_members = None  # None → AIFS-Single (det); int → AIFS-ENS sequential

storage_bucket = "aifs-modal-unibe"
initial_conditions_prefix = "aifs-ic-ifs"
initial_conditions_branch = "main"
outputs_prefix = "aifs-outputs-det"

marker_file = "data/interim/aifs_forecast_claudia_2025110700.done"

In [ ]:
# process params
start_date = dt.datetime.strptime(str(init_date_str), "%Y%m%d%H").replace(tzinfo=dt.UTC)
outputs_branch = f"{storm_name}-{init_date_str}"
end_date = start_date + dt.timedelta(hours=lead_time)

# papermill cannot inject None — use 0 as sentinel for deterministic
if n_members == 0:
    n_members = None

print(f"Storm   : {storm_name.capitalize()}")
print(f"Init    : {start_date}")
print(f"End     : {end_date}  (T+{lead_time} h)")
print(f"Members : {'deterministic' if n_members is None else n_members}")
print(f"Branch  : {outputs_branch}")

In [ ]:
# ── Run forecast on Modal GPU ─────────────────────────────────────────────────
# n_members=None  → AIFS-Single checkpoint (deterministic, fastest)
# n_members=N     → AIFS-ENS checkpoint, N members run sequentially on one GPU
with app.run():
    _t0 = time.perf_counter()
    run_forecast.remote(
        start_date,
        storage_bucket,
        n_members=n_members,
        lead_time=lead_time,
        initial_conditions_prefix=initial_conditions_prefix,
        initial_conditions_branch=initial_conditions_branch,
        outputs_prefix=outputs_prefix,
        outputs_branch=outputs_branch,
        include_pressure_levels=False,
    )
    print(f"Forecast completed in {time.perf_counter() - _t0:.1f}s")

In [ ]:
pathlib.Path(marker_file).touch()
print(f"Marker written: {marker_file}")